In [1]:
%matplotlib qt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

In [42]:
df = pd.read_csv('Full Dataset.csv').drop(columns=['Unnamed: 0'])
len(df[df.isna().any(axis=1)])

188

In [43]:
# Changing columns names for more readability
df.columns = ['village_code', 'visit_month', 'spray_status', 'total_residents',
       'clinic_tests', 'total_rdts', 'avg_distance_to_health_facility_km',
       'incidence_per_1000', 'positivity_rate', 'dist_missing', 'rainfall',
       'rainfall_total', 'humidity', 'temp', 'max_temp', 'min_temp', 'temp_range',
       'prev_rainfall', 'prev2_rainfall', 'prev3_rainfall',
       'prev4_rainfall', 'prev_rainfall_total', 'prev2_rainfall_total',
       'prev3_rainfall_total', 'prev4_rainfall_total', 'prev_humidity',
       'prev2_humidity', 'prev3_humidity', 'prev4_humidity', 'prev_temp', 'prev2_temp',
       'prev3_temp', 'prev4_temp', 'prev_max_temp', 'prev2_max_temp',
       'prev3_max_temp', 'prev4_max_temp', 'prev_min_temp', 'prev2_min_temp',
       'prev3_min_temp', 'prev4_min_temp', 'prev_temp_range', 'prev2_temp_range',
       'prev3_temp_range', 'prev4_temp_range', 'prev_incidence_per_1000']

In [44]:
months = df['visit_month'].unique()
months.sort()
lengths = []
for i, x in enumerate(months):
    lengths.append(len(df[df['visit_month'] == x]))
# This shows a very average distribution. No imbalances. Nice

In [45]:
df.loc[df['dist_missing'], 'avg_distance_to_health_facility_km'] = 1.4 * df['avg_distance_to_health_facility_km'].max()

In [46]:
df['avg_distance_to_health_facility_km'].max()

np.float64(42.14)

In [47]:
parameters = ['incidence_per_1000', 'avg_distance_to_health_facility_km', 'positivity_rate', 'spray_status']
weights = [4, 1.5, 1.5, -.3]

In [48]:
temp_df = df[df['visit_month'] <= '2017-01-01'].copy()
temp_df['visit_month'] = pd.to_datetime(temp_df['visit_month'])
df = df[df['visit_month'] > '2017-01-01'].copy()
mpi_df = pd.DataFrame({'village_code': [],'incidence_per_1000': [], 'avg_distance_to_health_facility_km': [], 
                       'positivity_rate': [], 'spray_status': []})
mpi_df['village_code'] = temp_df['village_code'].unique()
# I wasn't sure how to calculate the mean and create a new column at once so I created a (4, -1) array to contain the values
values = [[], [], [], []]
parameters = ['incidence_per_1000', 'avg_distance_to_health_facility_km', 'positivity_rate', 'spray_status']
for index, parameter in enumerate(parameters):
    for village in temp_df['village_code'].unique():
        values[index].append(temp_df.loc[temp_df['village_code'] == village, parameter].describe()['mean'])
# Now we finally create the column and fill them with the average values
for index, parameter in enumerate(parameters):
    mpi_df[parameter] = values[index]
mpi_df = mpi_df[mpi_df['incidence_per_1000'] < 500] # This clip is necessary

In [49]:
mpi_df

,village_code,incidence_per_1000,avg_distance_to_health_facility_km,positivity_rate,spray_status
0,7001,34.555,42.14,0.630,1.0
1,7002,2.445,7.60,0.365,1.0
2,7003,4.325,1.80,0.510,0.0
3,7004,9.015,0.70,0.565,1.0
4,7005,7.665,0.90,0.525,1.0
...,...,...,...,...,...
170,7233,9.375,5.30,0.380,1.0
171,7236,103.200,7.00,0.720,0.0
172,7237,35.600,6.00,0.830,0.0
173,7238,153.225,7.00,0.700,1.0


In [50]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
for x in parameters:
    mpi_df[x] = scaler.fit_transform(np.array(mpi_df[x]).reshape(-1, 1))

In [51]:
mpi_df['mpi'] = weights[0] * mpi_df['incidence_per_1000'] + weights[1] * mpi_df['avg_distance_to_health_facility_km'] \
               + weights[2] * mpi_df['positivity_rate'] + weights[3] * mpi_df['spray_status']

In [52]:
for x in mpi_df['village_code']:
    df.loc[df['village_code'] == x, 'mpi'] = round(mpi_df.loc[mpi_df['village_code'] == x, 'mpi'].iloc[0], 3)

In [53]:
df = df[~df.isna().any(axis=1)]

In [54]:
village_codes = df['village_code'].unique()
village_codes.sort()
split_point_village = int(len(village_codes) * .8)
village_bool = df['village_code'] <= village_codes[split_point_village]
len(df[village_bool]), len(df[~village_bool])

(2789, 660)

In [55]:
dates = df['visit_month'].unique().tolist()
dates.sort()
split_point_date = int(len('visit_month') * .6)
month_bool = df['visit_month'] >= dates[split_point_date]
# I made these changes from village_bool because here the mean > median
len(df[~month_bool]), len(df[month_bool])
dates[split_point_date] > '2016-09-01'

True

In [56]:
def chrono(chrono_bool):
    """
    Function for specifying which boolean to use for the (chronological or spatial) split
    """
    return month_bool if chrono_bool else village_bool

In [57]:
# 80% / 20% chronological split
target_columns = ['incidence_per_1000'] # These are the columns our model tries to predict
# And these are poor or useless columns that carry information about which villages have suffered epidemics before
# Causing it to overfit by predicting historical average
useless_columns = ['total_residents', 'visit_month', 'village_code', 'avg_distance_to_health_facility_km', 'clinic_tests',
                   'rainfall', 'rainfall_total', 'total_rdts', 'positivity_rate'] \
                + ['prev_rainfall', 'prev_rainfall_total', 'prev_min_temp', 'prev_max_temp', 'prev_temp', 'prev_humidity'] \
                + ['prev2_rainfall_total', 'prev3_rainfall_total', 'prev4_rainfall_total', 'prev_temp_range', 'prev2_temp_range',\
                   'prev3_temp_range', 'prev4_temp_range', 'temp_range']
                 # + ['humidity', 'temp', 'max_temp', 'min_temp', 'temp_range', 'total_rdts']
# Train data becomes the majority (80%)
X_train = df[chrono(1)]
y_train = X_train[target_columns]
# Test data becomes the minority (20%)
X_test = df[~chrono(1)]
y_test = X_test[target_columns]
# We finally drop the target columns from the X data
X_train = X_train.drop(columns=target_columns + useless_columns) # We also don't need 'visit_month' anymore it isn't important in training
X_test = X_test.drop(columns=target_columns + useless_columns)

In [58]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [59]:
model = RandomForestRegressor(max_depth=10, random_state=42).fit(X_train_scaled, np.ravel(y_train))

In [60]:
linear = LinearRegression().fit(X_train_scaled, y_train)

In [61]:
dt = DecisionTreeRegressor().fit(X_train_scaled, y_train)

In [62]:
knn = KNeighborsRegressor().fit(X_train_scaled, y_train)

In [63]:
ridge = GridSearchCV(
    Ridge(),
    param_grid={ 'alpha': np.linspace(100, 1000, 10)},
    cv=5,
    n_jobs=2,
)
ridge.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=Ridge(), n_jobs=2,
             param_grid={'alpha': array([ 100.,  200.,  300.,  400.,  500.,  600.,  700.,  800.,  900.,
       1000.])})

In [64]:
lasso = GridSearchCV(
    Lasso(),
    param_grid={ 'alpha': [.1, .2, .3, .4, .5] },
    cv=5,
    n_jobs=2,
)
lasso.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=Lasso(), n_jobs=2,
             param_grid={'alpha': [0.1, 0.2, 0.3, 0.4, 0.5]})

In [65]:
elasticnet = GridSearchCV(
    ElasticNet(),
    param_grid={ 'alpha': [.001, .01, .1, 1, 100, 1000], 'l1_ratio': [.1, .3, .5, .7, .9] },
    cv=5,
    n_jobs=2,
)
elasticnet.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=ElasticNet(), n_jobs=2,
             param_grid={'alpha': [0.001, 0.01, 0.1, 1, 100, 1000],
                         'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]})

In [66]:
bv1_pred = []
for x in range(1, len(df['incidence_per_1000']) + 1):
    bv1_pred.append(df['incidence_per_1000'].mean())
bv1_pred = np.array(bv1_pred)

In [67]:
metrics_bv1 = [100 * r2_score(bv1_pred, df['incidence_per_1000']),\
               mean_absolute_error(bv1_pred, df['incidence_per_1000']), \
               root_mean_squared_error(bv1_pred, df['incidence_per_1000'])]

In [68]:
metrics_bv2 = knn_metrics = [100 * r2_score(df['prev_incidence_per_1000'], df['incidence_per_1000']),\
                             mean_absolute_error(df['prev_incidence_per_1000'], df['incidence_per_1000']), \
                             root_mean_squared_error(df['prev_incidence_per_1000'], df['incidence_per_1000'])]

In [69]:
# Note that this is a way less stable and, somewhat, less reliable of a model because it loses all of its predictive power on a chonological split.
knn_metrics = [100 * r2_score(y_test, knn.predict(X_test_scaled)), mean_absolute_error(y_test, knn.predict(X_test_scaled)), \
           root_mean_squared_error(y_test, knn.predict(X_test_scaled))]

In [70]:
# Not that this is a way less stable and, somewhat, less reliable of a model because it loses all of its predictive power on a chonological split.
linear_metrics = [100 * r2_score(y_test, linear.predict(X_test_scaled)), mean_absolute_error(y_test, linear.predict(X_test_scaled)), \
           root_mean_squared_error(y_test, linear.predict(X_test_scaled))]

In [71]:
# This one is way more reliable
rf_metrics = [100 * r2_score(y_test, model.predict(X_test_scaled)), mean_absolute_error(y_test, model.predict(X_test_scaled)), \
          root_mean_squared_error(y_test, model.predict(X_test_scaled))]

In [72]:
metrics_dt = [100 * r2_score(y_test, dt.predict(X_test_scaled)), mean_absolute_error(y_test, dt.predict(X_test_scaled)), \
             root_mean_squared_error(y_test, dt.predict(X_test_scaled))]

In [73]:
metrics_ridge = [100 * r2_score(y_test, ridge.predict(X_test_scaled)), mean_absolute_error(y_test, ridge.predict(X_test_scaled)), \
                 root_mean_squared_error(y_test, ridge.predict(X_test_scaled))]

In [74]:
metrics_lasso = [100 * r2_score(y_test, lasso.predict(X_test_scaled)), mean_absolute_error(y_test, lasso.predict(X_test_scaled)), \
                 root_mean_squared_error(y_test, lasso.predict(X_test_scaled))]

In [75]:
metrics_enet = [100 * r2_score(y_test, elasticnet.predict(X_test_scaled)), mean_absolute_error(y_test, elasticnet.predict(X_test_scaled)), \
                root_mean_squared_error(y_test, elasticnet.predict(X_test_scaled))]

In [76]:
print(f"""
KNN: {knn_metrics}\nLR: {linear_metrics}\nRF: {rf_metrics}\nDT: {metrics_dt}\nRidge: {metrics_ridge}\nLasso: {metrics_lasso}\nElasticNet: {metrics_enet}
""")


KNN: [61.72549400642493, 42.929895615866386, 75.73818837953951]
LR: [67.13475215229307, 42.3995290019699, 70.1824575559387]
RF: [68.63433151241107, 40.66549994213917, 68.56261759240209]
DT: [18.53803606810859, 64.3076617954071, 110.49370946774094]
Ridge: [69.4613777349726, 38.42481072878855, 67.65265358492736]
Lasso: [69.71040288343661, 37.87765159366914, 67.37625444728499]
ElasticNet: [69.23687511600359, 38.058251221704126, 67.90087019011173]



In [77]:
# Sample data
data = [
    knn_metrics,
    rf_metrics,
    metrics_dt,
    linear_metrics,
    metrics_ridge,
    metrics_lasso,
    metrics_enet
]

# New labels for the groups (models)
group_labels = ['k-Nearest Neighbors', 'Random Forest', 'Decision Trees', 'Linear Regression', 'Ridge Regression', 'Lasso Regression', 'ElasticNet']
# New labels for the elements (metrics)
element_labels = ['R2 * 100 (0 - 100)', 'MAE', 'RMSE']
# Set the width of the bars
bar_width = 0.25
# Set the positions of the bar groups on the x-axis
x_positions = np.arange(len(group_labels))
# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the bars for each of the three elements in all groups
for i in range(3):
    values = [array[i] for array in data]
    ax.bar(x_positions + i * bar_width, values, bar_width, label=element_labels[i])

# Set the title and labels
ax.set_title('Model Performance Metrics Comparison')
ax.set_xlabel('Models')
ax.set_ylabel('Metric Value')

# Set the x-axis tick positions and labels
ax.set_xticks(x_positions + bar_width)
ax.set_xticklabels(group_labels, rotation=45, ha='right', fontsize=7)
# Add a legend to distinguish the elements
ax.legend()

plt.show()

In [110]:
coef_df = pd.DataFrame({
    'features': X_train.columns,
    'coefs': lasso.best_estimator_.coef_
})
coef_df.sort_values('coefs')
# plt.figure(figsize=(10, 6))
# plt.xlabel('Features'); plt.ylabel('Importances')
# plt.bar(coef_df['features'], coef_df['coefs'])
# plt.xticks(rotation=45, ha='right', fontsize=7)
coef_df[coef_df['coefs'] == 0]['features']

2           humidity
3               temp
5           min_temp
9     prev2_humidity
10    prev3_humidity
11    prev4_humidity
12        prev2_temp
13        prev3_temp
14        prev4_temp
18    prev2_min_temp
Name: features, dtype: object

In [111]:
# Let's plot the feature importance and HOPE it's not overfitting
importances = model.feature_importances_
plt.figure(figsize=(20, 8))
plt.bar(X_test.columns, importances)
plt.xlabel('Features'); plt.ylabel('Importances')
plt.title('Feature Importance Bar Chart')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [80]:
knn_o = joblib.load('knn.pkl')
linear_o = joblib.load('linear.pkl')
dt_o = joblib.load('dt.pkl')
rf_o = joblib.load('rf.pkl')
ridge_o = joblib.load('ridge.pkl')
lasso_o = joblib.load('lasso.pkl')
enet_o = joblib.load('elasticnet.pkl')

In [81]:
# -----------------------------
# Model names
models = ["Decision Tree", "KNN", "Linear Regression", "ElasticNet", "Ridge Regression", "Lasso Regression", "Random Forest"]

# -----------------------------
# Metrics
# Baselines
r2_baseline = [None, None, None, None, None, None, None]
mae_baseline = [None, None, None, None, None, None, None]
rmse_baseline = [None, None, None, None, None, None, None]

# Without MPI
r2_no_mpi = [0.193, 0.498, 0.593, 0.619, 0.621, 0.627, 0.628]
mae_no_mpi = [63.51, 50.51, 44.07, 41.29, 41.50, 40.93, 43.25]
rmse_no_mpi = [110.47, 87.17, 78.46, 75.93, 75.66, 75.06, 74.94]

# With MPI
r2_mpi = [0.216, 0.617, 0.671, 0.692, 0.694, 0.697, 0.686]
mae_mpi = [63.30, 42.93, 42.40, 38.06, 38.42, 37.88, 40.67]
rmse_mpi = [108.37, 75.74, 70.18, 67.90, 67.65, 67.38, 68.56]

# -----------------------------
# Plot all metrics in one figure
x = np.arange(len(models))
width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(18,6))

# R²
# axes[0].bar(x - width, [v if v is not None else 0 for v in r2_baseline], width, color='red', label='Baseline')
axes[0].bar(x, r2_no_mpi, width, color='blue', label='Without MPI')
axes[0].bar(x + width, r2_mpi, width, color='green', label='With MPI')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=45, ha='right')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Comparison')
axes[0].legend()

# MAE
# axes[1].bar(x - width, [v if v is not None else 0 for v in mae_baseline], width, color='red', label='Baseline')
axes[1].bar(x, mae_no_mpi, width, color='blue', label='Without MPI')
axes[1].bar(x + width, mae_mpi, width, color='green', label='With MPI')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=45, ha='right')
axes[1].set_ylabel('MAE')
axes[1].set_title('MAE Comparison')

# RMSE
# axes[2].bar(x - width, [v if v is not None else 0 for v in rmse_baseline], width, color='red', label='Baseline')
axes[2].bar(x, rmse_no_mpi, width, color='blue', label='Without MPI')
axes[2].bar(x + width, rmse_mpi, width, color='green', label='With MPI')
axes[2].set_xticks(x)
axes[2].set_xticklabels(models, rotation=45, ha='right')
axes[2].set_ylabel('RMSE')
axes[2].set_title('RMSE Comparison')

plt.suptitle('Model Performance Comparison: Baseline, Without MPI, With MPI', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [101]:
plt.scatter(np.log1p(y_test), np.log1p(lasso.predict(X_test_scaled)), s=4); plt.plot([0, 8], [0, 8], 'r')

C:\Users\User\AppData\Local\Temp\ipykernel_1148\3606464777.py:1: RuntimeWarning: invalid value encountered in log1p
  plt.scatter(np.log1p(y_test), np.log1p(lasso.predict(X_test_scaled)), s=4); plt.plot([0, 8], [0, 8], 'r')
